In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

from skimage import color, filters, exposure, morphology, measure
from skimage.feature import local_binary_pattern
from skimage.filters import gabor_kernel
from skimage.morphology import (
    binary_opening, binary_closing, binary_dilation, disk, remove_small_objects
)
from skimage.measure import label, regionprops
from scipy import ndimage as ndi
from scipy.stats import wasserstein_distance

print("All imports successful.")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_DIR  = Path("wood")
OUTPUT_DIR = Path("griffin_output")
OUTPUT_DIR.mkdir(exist_ok=True)

image_paths = sorted(INPUT_DIR.glob("*.png"))
print(f"Input  folder : {INPUT_DIR.resolve()}")
print(f"Output folder : {OUTPUT_DIR.resolve()}")
print(f"Images found  : {len(image_paths)}")
for p in image_paths:
    print(f"  {p.name}")

In [ ]:
# ── Gabor Filter Bank ──────────────────────────────────────────────────────────

def griffin_build_gabor_bank(frequencies=(0.1, 0.2, 0.35), n_orientations=6):
    """
    Build a bank of Gabor kernels at multiple frequencies and orientations.
    Each (frequency, orientation) pair is sensitive to a different texture pattern.
    Wood grain is highly directional, so defects show as outliers across orientations.
    """
    kernels = []
    for freq in frequencies:
        for theta_idx in range(n_orientations):
            theta = theta_idx / n_orientations * np.pi
            kernel = gabor_kernel(freq, theta=theta, sigma_x=3, sigma_y=3)
            kernels.append((freq, theta, kernel))
    return kernels


def griffin_gabor_energy(gray, kernels):
    """
    Compute the Gabor energy response for each kernel and stack into a
    (H, W, n_kernels) array. Energy = sqrt(real^2 + imag^2).
    """
    energy_maps = []
    for _, _, kernel in kernels:
        real = ndi.convolve(gray, kernel.real, mode='reflect')
        imag = ndi.convolve(gray, kernel.imag, mode='reflect')
        energy_maps.append(np.sqrt(real**2 + imag**2))
    return np.stack(energy_maps, axis=-1)  # (H, W, n_kernels)


GABOR_BANK = griffin_build_gabor_bank()
print(f"Gabor bank built: {len(GABOR_BANK)} kernels")

In [ ]:
# ── LBP Texture Analysis ───────────────────────────────────────────────────────

LBP_RADIUS   = 3
LBP_N_POINTS = 8 * LBP_RADIUS   # 24
LBP_N_BINS   = LBP_N_POINTS + 2  # uniform LBP bins


def griffin_compute_lbp(gray):
    """Compute uniform LBP image."""
    return local_binary_pattern(gray, LBP_N_POINTS, LBP_RADIUS, method='uniform')


def griffin_lbp_histogram(lbp_patch):
    """Normalised LBP histogram for a patch."""
    hist, _ = np.histogram(lbp_patch, bins=LBP_N_BINS,
                           range=(0, LBP_N_BINS), density=True)
    return hist


def griffin_lbp_deviation_map(lbp, patch_size=32, stride=8):
    """
    Slide a window over the LBP image and compute the Wasserstein distance
    between each patch's histogram and the global (background) histogram.
    Regions that deviate strongly from the dominant wood-grain texture get
    a high score.
    """
    H, W = lbp.shape
    global_hist = griffin_lbp_histogram(lbp)

    score_map = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)

    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patch = lbp[y:y+patch_size, x:x+patch_size]
            patch_hist = griffin_lbp_histogram(patch)
            dist = wasserstein_distance(global_hist, patch_hist)
            score_map[y:y+patch_size, x:x+patch_size] += dist
            count_map[y:y+patch_size, x:x+patch_size] += 1

    count_map[count_map == 0] = 1
    score_map /= count_map
    return score_map


print("LBP functions defined.")

In [ ]:
# ── Anomaly Score & Masking ────────────────────────────────────────────────────

def griffin_gabor_anomaly_map(energy_stack):
    """
    For each pixel, compute the std-dev of its Gabor energy vector across all
    kernels, then smooth. High std-dev = inconsistent response across
    orientations/frequencies = likely a defect breaking the grain pattern.
    """
    # Normalise each channel to [0,1] so frequencies don't dominate each other
    norm_stack = energy_stack.copy()
    for k in range(energy_stack.shape[-1]):
        ch = energy_stack[..., k]
        rng = ch.max() - ch.min()
        norm_stack[..., k] = (ch - ch.min()) / (rng + 1e-8)

    # Pixel-wise coefficient of variation across the filter bank
    mean_e = norm_stack.mean(axis=-1)
    std_e  = norm_stack.std(axis=-1)
    cv_map = std_e / (mean_e + 1e-8)

    # Smooth to suppress single-pixel noise
    cv_map = ndi.gaussian_filter(cv_map, sigma=3)
    return cv_map


def griffin_combine_scores(gabor_map, lbp_map, gabor_weight=0.55, lbp_weight=0.45):
    """Normalise both maps to [0,1] and compute a weighted sum."""
    def norm01(m):
        mn, mx = m.min(), m.max()
        return (m - mn) / (mx - mn + 1e-8)

    return gabor_weight * norm01(gabor_map) + lbp_weight * norm01(lbp_map)


def griffin_threshold_and_clean(score_map, percentile=92,
                                 min_area=120, close_radius=5, dilate_radius=3):
    """
    Threshold the anomaly score at `percentile`, then:
    - morphological closing fills small gaps
    - removes tiny noise regions
    - slight dilation to make boundaries visible
    """
    thresh = np.percentile(score_map, percentile)
    binary = score_map > thresh

    binary = binary_closing(binary, disk(close_radius))
    binary = remove_small_objects(binary, min_size=min_area)
    binary = binary_dilation(binary, disk(dilate_radius))
    return binary


print("Anomaly scoring functions defined.")

In [ ]:
# ── Visualisation ──────────────────────────────────────────────────────────────

def griffin_overlay(img_rgb, mask, color=(1.0, 0.15, 0.15), alpha=0.45):
    """Red semi-transparent overlay on detected defect regions."""
    overlay = img_rgb.copy().astype(np.float64) / 255.0
    for c, val in enumerate(color):
        overlay[..., c] = np.where(mask, alpha * val + (1 - alpha) * overlay[..., c],
                                          overlay[..., c])
    return np.clip(overlay, 0, 1)


def griffin_visualise_and_save(img_rgb, gray, score_map, mask, lbp_img,
                                image_name, output_dir):
    """5-panel figure: original | LBP | anomaly score | mask | overlay."""
    fig, axes = plt.subplots(1, 5, figsize=(26, 5))
    fig.suptitle(f"Griffin Texture Defect Detection — {image_name}",
                 fontsize=13, fontweight='bold', y=1.01)

    panels = [
        (img_rgb,                           "Original",       'gray'),
        (lbp_img,                           "LBP Texture",    'inferno'),
        (score_map,                         "Anomaly Score",  'hot'),
        (mask.astype(np.uint8) * 255,       "Defect Mask",    'gray'),
        (griffin_overlay(img_rgb, mask),    "Overlay",        None),
    ]

    for ax, (data, title, cmap) in zip(axes, panels):
        if cmap:
            ax.imshow(data, cmap=cmap)
        else:
            ax.imshow(data)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    stem = Path(image_name).stem
    out_path = output_dir / f"griffin_{stem}_result.png"
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return out_path


print("Visualisation functions defined.")

In [ ]:
# ── Full Detection Pipeline ────────────────────────────────────────────────────

def griffin_detect_defects(image_path, gabor_bank=GABOR_BANK,
                            percentile=92, min_area=120):
    """
    End-to-end defect detection for a single wood image.

    Steps
    -----
    1. Load image and convert to grayscale (CLAHE-enhanced for contrast).
    2. Apply Gabor filter bank → compute per-pixel energy coefficient of variation.
    3. Compute LBP → slide-window Wasserstein deviation map.
    4. Fuse scores, threshold, morphologically clean.
    5. Return mask + intermediate maps for visualisation.
    """
    # 1. Load
    img_pil = Image.open(image_path).convert('RGB')
    img_rgb = np.array(img_pil)
    gray_raw = color.rgb2gray(img_rgb)

    # Enhance contrast so subtle defects are not washed out
    gray = exposure.equalize_adapthist(gray_raw, clip_limit=0.025)

    # 2. Gabor energy anomaly
    energy_stack = griffin_gabor_energy(gray, gabor_bank)
    gabor_anom   = griffin_gabor_anomaly_map(energy_stack)

    # 3. LBP deviation
    lbp_img    = griffin_compute_lbp(gray)
    lbp_anom   = griffin_lbp_deviation_map(lbp_img)

    # 4. Fuse & threshold
    combined   = griffin_combine_scores(gabor_anom, lbp_anom)
    mask       = griffin_threshold_and_clean(combined, percentile=percentile,
                                             min_area=min_area)

    return img_rgb, gray, combined, mask, lbp_img


print("Detection pipeline defined.")

In [ ]:
# ── Region Stats Helper ────────────────────────────────────────────────────────

def griffin_region_stats(mask, image_name):
    """
    Label connected defect regions and print summary statistics:
    count, area, bounding-box, and a crude shape descriptor.
    """
    labeled = label(mask)
    props   = regionprops(labeled)

    print(f"\n{'─'*55}")
    print(f"  {image_name}  — {len(props)} defect region(s) detected")
    print(f"{'─'*55}")
    for i, prop in enumerate(props, 1):
        ar = prop.major_axis_length / (prop.minor_axis_length + 1e-3)
        shape = "crack/scratch" if ar > 4 else ("knot/hole" if prop.area > 800 else "speck")
        print(f"  Region {i:2d}: area={prop.area:6d}px  "
              f"bbox={tuple(int(v) for v in prop.bbox)}  "
              f"aspect={ar:.1f}  → {shape}")
    return props


print("Stats helper defined.")

In [ ]:
# ── Run on All Images ──────────────────────────────────────────────────────────

all_results = {}  # image_name -> {mask, props, score_map}

for img_path in image_paths:
    print(f"\nProcessing {img_path.name} …", end=" ", flush=True)

    try:
        img_rgb, gray, score_map, mask, lbp_img = griffin_detect_defects(img_path)
        out_path = griffin_visualise_and_save(
            img_rgb, gray, score_map, mask, lbp_img,
            img_path.name, OUTPUT_DIR
        )
        props = griffin_region_stats(mask, img_path.name)
        all_results[img_path.name] = {
            'n_defects' : len(props),
            'total_area': sum(p.area for p in props),
            'output'    : str(out_path),
        }
        print(f"→ saved {out_path.name}")
    except Exception as e:
        print(f"ERROR: {e}")

print("\n✓ Done. All results in griffin_output/")

In [ ]:
# ── Summary Table ──────────────────────────────────────────────────────────────

print(f"\n{'Image':<12} {'Defect Regions':>16} {'Total Defect Area (px)':>24}")
print("─" * 54)
for name, info in sorted(all_results.items()):
    print(f"{name:<12} {info['n_defects']:>16} {info['total_area']:>24}")

total_detections = sum(v['n_defects'] for v in all_results.values())
print("─" * 54)
print(f"{'TOTAL':<12} {total_detections:>16}")